Pre model testing

In [1]:
import torch
from facenet_pytorch import MTCNN
from mobilefacenet import MobileFaceNet
import cv2
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mtcnn = MTCNN(image_size=112, margin=20, device=device, post_process=True)

model = MobileFaceNet()
model.load_state_dict(torch.load('global_model_v0.pth', map_location=device))
model.to(device).eval()
print("[INFO] Model pre-trained 'global_model_v0.pth' berhasil dimuat.")

[INFO] Model pre-trained 'global_model_v0.pth' berhasil dimuat.


In [2]:
import torchvision.transforms as T
from PIL import Image

augmentation = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2), # Random brightness 
    T.RandomRotation(degrees=15),                # Random angle
    T.RandomHorizontalFlip(p=0.5)
])

final_resize = T.Resize((112, 96)) 

ROOT_DATA = 'datasets' 
OUTPUT_DATA = 'datasets_processed'

def run_batch_preprocessing(source_dir):
    
    if not os.path.exists(OUTPUT_DATA):
        os.makedirs(OUTPUT_DATA)

    for nrp_folder in os.listdir(source_dir):
        path_nrp = os.path.join(source_dir, nrp_folder)
        if not os.path.isdir(path_nrp): continue
        
        # Buat folder output per NRP 
        save_dir = os.path.join(OUTPUT_DATA, nrp_folder)
        os.makedirs(save_dir, exist_ok=True)
        
        print(f"[PROCESS] Mengolah data NRP: {nrp_folder}")
        
        for img_name in os.listdir(path_nrp):
            img_path = os.path.join(path_nrp, img_name)
            try:
                img = Image.open(img_path).convert('RGB')
                
                face = mtcnn(img)
                
                if face is not None:
                    # Konversi tensor kembali ke PIL Image
                    face_img = T.ToPILImage()(face * 0.5 + 0.5)
                    
                    face_img = final_resize(face_img)
                    
                    # Simpan hasil asli
                    face_img.save(os.path.join(save_dir, img_name))
                    
                    # Simpan hasil augmentasi untuk memperkaya dataset
                    aug_face = augmentation(face_img)
                    aug_face.save(os.path.join(save_dir, f"aug_{img_name}"))
            except Exception as e:
                print(f"[SKIP] Gagal memproses {img_name}: {e}")

run_batch_preprocessing(ROOT_DATA)
print("\n[DONE] Semua NRP telah diproses dan di-resize ke 112x96")

[PROCESS] Mengolah data NRP: 241044
[PROCESS] Mengolah data NRP: 241026
[PROCESS] Mengolah data NRP: 241034
[PROCESS] Mengolah data NRP: 231086
[PROCESS] Mengolah data NRP: 241032
[PROCESS] Mengolah data NRP: 221072
[PROCESS] Mengolah data NRP: 241093
[PROCESS] Mengolah data NRP: 231035
[PROCESS] Mengolah data NRP: 241048
[PROCESS] Mengolah data NRP: 241079
[PROCESS] Mengolah data NRP: 241051
[PROCESS] Mengolah data NRP: 241065
[PROCESS] Mengolah data NRP: 231030
[PROCESS] Mengolah data NRP: 221050
[PROCESS] Mengolah data NRP: 231009
[PROCESS] Mengolah data NRP: 241068
[PROCESS] Mengolah data NRP: 241115
[PROCESS] Mengolah data NRP: 221021
[PROCESS] Mengolah data NRP: 241040
[PROCESS] Mengolah data NRP: 241089
[PROCESS] Mengolah data NRP: 241039
[PROCESS] Mengolah data NRP: 241085
[PROCESS] Mengolah data NRP: 241008
[PROCESS] Mengolah data NRP: 241022
[PROCESS] Mengolah data NRP: 241072
[PROCESS] Mengolah data NRP: 241102
[PROCESS] Mengolah data NRP: 231027
[PROCESS] Mengolah data NRP:

In [3]:
processed_dir = 'datasets_processed'
stats = {folder: len(os.listdir(os.path.join(processed_dir, folder))) 
         for folder in os.listdir(processed_dir) if os.path.isdir(os.path.join(processed_dir, folder))}

print("Statistik Data Per NRP:")
for nrp, count in stats.items():
    print(f"NRP {nrp}: {count} gambar")

# Cek apakah ada yang di bawah target 20 foto per individu (sesuai proposal 3.1.1.1)
target = 20
under_target = [nrp for nrp, count in stats.items() if count < target]
if under_target:
    print(f"\n[PERINGATAN] NRP berikut masih di bawah target {target} foto: {under_target}")

Statistik Data Per NRP:
NRP 241044: 64 gambar
NRP 241026: 98 gambar
NRP 241034: 214 gambar
NRP 231086: 154 gambar
NRP 241032: 142 gambar
NRP 221072: 182 gambar
NRP 241093: 46 gambar
NRP 231035: 282 gambar
NRP 241048: 158 gambar
NRP 241079: 400 gambar
NRP 241051: 298 gambar
NRP 241065: 166 gambar
NRP 231030: 188 gambar
NRP 221050: 154 gambar
NRP 231009: 470 gambar
NRP 241068: 154 gambar
NRP 241115: 210 gambar
NRP 221021: 148 gambar
NRP 241040: 144 gambar
NRP 241089: 154 gambar
NRP 241039: 262 gambar
NRP 241085: 152 gambar
NRP 241008: 166 gambar
NRP 241022: 246 gambar
NRP 241072: 128 gambar
NRP 241102: 90 gambar
NRP 231027: 226 gambar
NRP 241094: 164 gambar
NRP 241095: 130 gambar
NRP 231014: 422 gambar
NRP 241096: 304 gambar
NRP 241036: 200 gambar
NRP 221063: 216 gambar
NRP 231028: 154 gambar


In [4]:
import random
import shutil

INPUT_DIR = 'datasets_processed'
BALANCED_DIR = 'datasets_balanced'
TARGET_COUNT = 100 

augment_transform = T.Compose([
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomRotation(degrees=20),
    T.RandomHorizontalFlip(p=0.5)
])

def smart_balance():
    if not os.path.exists(BALANCED_DIR):
        os.makedirs(BALANCED_DIR)

    for nrp_folder in os.listdir(INPUT_DIR):
        src_path = os.path.join(INPUT_DIR, nrp_folder)
        dst_path = os.path.join(BALANCED_DIR, nrp_folder)
        if not os.path.isdir(src_path): continue
        
        os.makedirs(dst_path, exist_ok=True)
        
        # Pisahkan file asli dan file yang sudah kena augmentasi sebelumnya
        all_files = [f for f in os.listdir(src_path) if f.endswith(('.jpg', '.png'))]
        clean_images = [f for f in all_files if not f.startswith('aug_')]
        
        count_now = len(all_files)
        print(f"[INFO] Balancing {nrp_folder}: {count_now} -> {TARGET_COUNT}")

        # Salin semua file yang sudah ada ke folder balanced dulu
        for f in all_files:
            shutil.copy(os.path.join(src_path, f), os.path.join(dst_path, f))

        if count_now < TARGET_COUNT:
            # KASUS: KURANG DATA
            # Kita hanya ambil dari 'clean_images' sebagai bahan baku tambahan
            needed = TARGET_COUNT - count_now
            for i in range(needed):
                base_img_name = random.choice(clean_images)
                img = Image.open(os.path.join(src_path, base_img_name))
                
                # Apply transform baru dari bahan asli
                aug_img = augment_transform(img)
                aug_img.save(os.path.join(dst_path, f"extra_aug_{i:03d}_{base_img_name}"))
                
        elif count_now > TARGET_COUNT:
            # KASUS: KELEBIHAN DATA (Undersampling)
            # Hapus semua di dst_path dan pilih 100 secara acak dari all_files
            for f in os.listdir(dst_path): os.remove(os.path.join(dst_path, f))
            selected = random.sample(all_files, TARGET_COUNT)
            for img_name in selected:
                shutil.copy(os.path.join(src_path, img_name), os.path.join(dst_path, img_name))

smart_balance()
print("\n[DONE] Dataset Balanced")

[INFO] Balancing 241044: 64 -> 100
[INFO] Balancing 241026: 98 -> 100
[INFO] Balancing 241034: 214 -> 100
[INFO] Balancing 231086: 154 -> 100
[INFO] Balancing 241032: 142 -> 100
[INFO] Balancing 221072: 182 -> 100
[INFO] Balancing 241093: 46 -> 100
[INFO] Balancing 231035: 282 -> 100
[INFO] Balancing 241048: 158 -> 100
[INFO] Balancing 241079: 400 -> 100
[INFO] Balancing 241051: 298 -> 100
[INFO] Balancing 241065: 166 -> 100
[INFO] Balancing 231030: 188 -> 100
[INFO] Balancing 221050: 154 -> 100
[INFO] Balancing 231009: 470 -> 100
[INFO] Balancing 241068: 154 -> 100
[INFO] Balancing 241115: 210 -> 100
[INFO] Balancing 221021: 148 -> 100
[INFO] Balancing 241040: 144 -> 100
[INFO] Balancing 241089: 154 -> 100
[INFO] Balancing 241039: 262 -> 100
[INFO] Balancing 241085: 152 -> 100
[INFO] Balancing 241008: 166 -> 100
[INFO] Balancing 241022: 246 -> 100
[INFO] Balancing 241072: 128 -> 100
[INFO] Balancing 241102: 90 -> 100
[INFO] Balancing 231027: 226 -> 100
[INFO] Balancing 241094: 164 -> 

In [5]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from mobilefacenet import MobileFaceNet, ArcMarginProduct 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 20
LR = 0.001
NUM_CLASSES = 34 # Mahasiswa

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = datasets.ImageFolder('datasets_balanced', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

model = MobileFaceNet().to(device)
model.load_state_dict(torch.load('global_model_v0.pth', map_location=device))

metric_fc = ArcMarginProduct(128, NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {'params': model.parameters()},
    {'params': metric_fc.parameters()}
], lr=LR)

print(f"[START] Memulai raining di")
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    correct = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        embeddings = model(images)
        outputs = metric_fc(embeddings, labels)
        
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / len(train_dataset)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss/len(train_loader):.4f} - Acc: {acc:.2f}%")

torch.save(model.state_dict(), 'test_model.pth')
print("\n[DONE] Baseline Model Berhasil Disimpan: 'test_model.pth'")

[START] Memulai raining di
Epoch [1/20] - Loss: 6.2331 - Acc: 43.01%
Epoch [2/20] - Loss: 0.3430 - Acc: 92.51%
Epoch [3/20] - Loss: 0.0531 - Acc: 98.69%
Epoch [4/20] - Loss: 0.0192 - Acc: 99.69%
Epoch [5/20] - Loss: 0.0099 - Acc: 99.77%
Epoch [6/20] - Loss: 0.0127 - Acc: 99.77%
Epoch [7/20] - Loss: 0.0492 - Acc: 98.71%
Epoch [8/20] - Loss: 0.0450 - Acc: 98.71%
Epoch [9/20] - Loss: 0.0515 - Acc: 98.89%
Epoch [10/20] - Loss: 0.0673 - Acc: 98.31%
Epoch [11/20] - Loss: 0.0408 - Acc: 98.89%
Epoch [12/20] - Loss: 0.0282 - Acc: 99.00%
Epoch [13/20] - Loss: 0.0330 - Acc: 99.23%
Epoch [14/20] - Loss: 0.0609 - Acc: 98.51%
Epoch [15/20] - Loss: 0.0299 - Acc: 99.14%
Epoch [16/20] - Loss: 0.0201 - Acc: 99.40%
Epoch [17/20] - Loss: 0.0078 - Acc: 99.77%
Epoch [18/20] - Loss: 0.0521 - Acc: 98.43%
Epoch [19/20] - Loss: 0.0171 - Acc: 99.40%
Epoch [20/20] - Loss: 0.0053 - Acc: 99.89%

[DONE] Baseline Model Berhasil Disimpan: 'test_model.pth'
